**Part 1 — Data Acquisition, Cleaning, and Exploratory Analysis**

In [ ]:
import pandas as pd
#Reading the dataset
df=pd.read_csv("https://raw.githubusercontent.com/fenago/datasets/master/Car%20details%20v3.csv")
#Printing data shape and data types
print(df.shape)
print(df.dtypes)
print(df.head(5))

In [ ]:
#Finding null count
null_count=df.isnull().sum()
print(f'Null Count:{null_count}')


In [ ]:
#Finding null percentage
null_percentage=((df.isnull().sum()/df.shape[0])*100)
print(f'Null Percentage:{null_percentage}')


In [ ]:
#Finding columns which has null values greater than 20%
col_with_null_values=null_percentage[null_percentage>20]
if len(col_with_null_values) == 0:
    print('No columns have null values greater than 20%.')
else:
    print(f'Columns with null values greater than 20%:{col_with_null_values}')

In [ ]:
#Converting datatypes of actual numeric columns from object to numeric type
df_clean=df.copy()
df_clean['mileage']=pd.to_numeric(df_clean['mileage'].str.replace(' kmpl','', regex=False), errors='coerce')
df_clean['engine']=pd.to_numeric(df_clean['engine'].str.replace(' CC','', regex=False), errors='coerce')
df_clean['max_power']=pd.to_numeric(df_clean['max_power'].str.replace(' bhp','', regex=False), errors='coerce')
print(df_clean.dtypes)

In [ ]:
#displaying labels of all numeric columns
numeric_columns = df_clean.select_dtypes(include=['int64', 'float64']).columns
# print(numeric_columns)
#Filling numeric cols with null percentage below 20 using median
for col in numeric_columns:
  null_percentage = (df_clean[col].isnull().sum() / df_clean.shape[0]) * 100
  if (null_percentage> 0) and (null_percentage < 20):
    col_median = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(col_median)
    print(f"Filled column '{col}' with its median value: {col_median}")

In [ ]:
#Finding duplicates and removing it
total_rows = df_clean.shape[0]
print(f"Total rows in the DataFrame: {total_rows}")
duplicates_count=df_clean.duplicated().sum()
print(f"Total duplicate rows in the DataFrame: {duplicates_count}")
drop_duplicates=df_clean.drop_duplicates()
print(f"Total rows after dropping duplicates: {drop_duplicates.shape[0]}")
print(f'Initial Null Percentage:{null_percentage}')
null_percentage_df_clean = (df_clean[col].isnull().sum() / df_clean.shape[0]) * 100
print(f'Null Percentage after dropping duplicates:{null_percentage_df_clean}')

In [ ]:
#Dropping torque column since its messy and not required for future purpose
df_clean=df_clean.drop(columns=['torque'])

In [ ]:
print(f'Original df:\n{df.isnull().sum()}\n')
print(f'Cleaned df:\n{df_clean.isnull().sum()}')

In [ ]:
#Memory usage before conversion of repetitive string columns to category
memory_before = df_clean.memory_usage(deep=True).sum()
print(f'Memory usage before conversion: {memory_before} bytes')
#Convert repetitive string columns to 'category' data type
categorical_cols = ['fuel', 'transmission', 'owner']
for col in categorical_cols:
    df_clean[col] = df_clean[col].astype('category')
#Memory usage after conversion of repetitive string columns to category
memory_after=df_clean.memory_usage(deep=True).sum()
print(f'Memory usage after conversion: {memory_after} bytes')

In [ ]:
#displaying statistics
df_clean.describe()

In [ ]:
#Finding highest skewed column in df_clean
core_cols = ['selling_price', 'km_driven', 'mileage', 'engine', 'max_power']
skew_values={}
for col in core_cols:
  skewness=df_clean[col].skew()
  skew_values[col]=skewness
  print(f"Skewness of '{col}': {skewness}")
highest_skew_col=max(skew_values)
print(f'Highest Skewed Column:{highest_skew_col}')

In [ ]:
#Detecting outliers
target_cols = ['selling_price', 'km_driven']
for col in target_cols:
  Q1=df[col].quantile(0.25)
  Q3=df[col].quantile(0.75)
  IQR=Q3-Q1
  lower_bound = Q1-1.5*IQR
  upper_bound = Q3+1.5*IQR
  outliers = df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)]
  outlier_count = len(outliers)
  print(f"\nColumn: '{col}'")
  print(f"Q1 (25th percentile): {Q1:,.2f}")
  print(f"Q3 (75th percentile): {Q3:,.2f}")
  print(f"IQR: {IQR:,.2f}")
  print(f"Lower Bound: {lower_bound:,.2f}")
  print(f"Upper Bound: {upper_bound:,.2f}")
  print(f"Number of Outliers Detected: {outlier_count} rows")


In [ ]:
#Creating a line plot
import matplotlib.pyplot as plt
#Grouping the data by 'year' and calculate the average kilometers driven instantly
timeline_data = df_clean.groupby('year')['km_driven'].mean()
plt.plot(timeline_data.index, timeline_data.values, marker='o', color='teal')
#Add clear titles and labels
plt.title('Vehicle Usage Trend: Average Kilometers Driven over the Years')
plt.xlabel('Manufacturing Year')
plt.ylabel('Average Kilometers Driven (KM)')
plt.show()

In [ ]:
#Creating the bar plot
import matplotlib.pyplot as plt
#Grouping the data by 'year' and calculate the mean selling price for each year
yearly_trends = df_clean.groupby('year')['selling_price'].mean().reset_index()
plt.figure(figsize=(8, 5))
plt.bar(yearly_trends['year'], yearly_trends['selling_price'],color='darkorange', linewidth=2.5)
#Add required titles and labels
plt.title('Market Trend: Average Vehicle Selling Price over Years', fontsize=14, fontweight='bold')
plt.xlabel('Manufacturing Year', fontsize=12)
plt.ylabel('Average Selling Price in Rupees', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
#Creating histogram
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(8, 5))
sns.histplot(df_clean['selling_price'], bins=20, color='crimson', kde=True)
#Add required title and labels
plt.title('Distribution of Selling Prices(Highly Skewed)')
plt.xlabel('Selling Price (in Rupees)')
plt.ylabel('Number of Vehicles (Count)')
plt.show()

In [ ]:
#Creating Scatter plot
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_clean, x='max_power', y='selling_price', color='dodgerblue', alpha=0.6)
#Add clear, required titles and labels
plt.title('Relationship Between Engine Max Power and Vehicle Selling Price')
plt.xlabel('Max Power (bhp)')
plt.ylabel('Selling Price (in Rupees)')
plt.show()

In [ ]:
#Creating box plot
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df_clean,
    x='fuel',
    y='km_driven',
    hue='fuel',
    legend=False,
    palette='Set3',
    width=0.8
)
#Add clear titles and labels required by your assignment
plt.title('Identifying Mileage Outliers Across Fuel Types')
plt.xlabel('Fuel Type')
plt.ylabel('Kilometers Driven (KM)')
plt.show()

Box Plot Distribution Analysis

- **Visible Differences in Median and Spread Across Categories:**
  - **Difference in Medians:** The median line inside the **Diesel** box sits visibly higher up the scale than the **Petrol** box. This confirms the market reality that diesel vehicles are systematically purchased for heavier, longer-distance usage.
  - **Difference in Spread and Outliers:** The **Diesel** box exhibits a taller vertical spread, indicating a wider variation in standard usage habits. Crucially, both categories display clear **individual outlier points** extending far above the upper whisker bound. These represent commercial cars with extreme mileage totals that deviate from typical consumer driving habits, showing exactly why the IQR method flagged them as statistical outliers.

In [ ]:
#Heatmp
import matplotlib.pyplot as plt
import seaborn as sns
#Selecting only the numeric columns to avoid any errors/warnings
numeric_df = df_clean.select_dtypes(include=['number'])
#Computing the correlation matrix
corr_matrix = numeric_df.corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix Heatmap of Numeric Variables')
plt.show()

In [ ]:
#Immputing strategy comparison
#Identify the two columns
skewed_cols = ['selling_price', 'km_driven']
for col in skewed_cols:
    col_mean = df_clean[col].mean()
    col_median = df_clean[col].median()
    print(f"Column: {col}")
    print(f"Mean  : {col_mean:.2f}")
    print(f"Median: {col_median:.2f}\n")
#Imputing missing values with the Median (since both are highly right-skewed)
for col in skewed_cols:
    median_value = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_value)
print(df_clean[skewed_cols].isnull().sum())

In [ ]:
#Spearman / Pearson Coefficients
import pandas as pd
import numpy as np

# 1. Isolate numeric columns
numeric_df = df_clean.select_dtypes(include=['number'])

# 2. Compute both correlation matrices
pearson_matrix = numeric_df.corr(method='pearson')
spearman_matrix = numeric_df.corr(method='spearman')

print("--- PEARSON CORRELATION MATRIX ---")
print(pearson_matrix.round(4))
print("\n--- SPEARMAN RANK CORRELATION MATRIX ---")
print(spearman_matrix.round(4))

# 3. Calculate absolute differences between the two matrices
diff_matrix = (spearman_matrix - pearson_matrix).abs()

# 4. Extract unique pairs (ignoring diagonal and duplicate matrix mirrors)
pairs = []
columns = numeric_df.columns
for i in range(len(columns)):
    for j in range(i + 1, len(columns)):
        col1, col2 = columns[i], columns[j]
        p_val = pearson_matrix.loc[col1, col2]
        s_val = spearman_matrix.loc[col1, col2]
        abs_diff = diff_matrix.loc[col1, col2]
        pairs.append({
            'Pair': f"{col1} & {col2}",
            'Pearson': p_val,
            'Spearman': s_val,
            '|Spearman - Pearson|': abs_diff
        })

# 5. Convert to dataframe and sort to find the top 3 highest differences
diff_df = pd.DataFrame(pairs).sort_values(by='|Spearman - Pearson|', ascending=False)

print("\n--- TOP 3 COLUMN PAIRS WITH LARGEST DIFFERENCE ---")
print(diff_df.head(3).to_string(index=False))

In [ ]:
#Computing grouped aggregation for transmission types on selling_price
grouped_summary = df_clean.groupby('transmission')['selling_price'].agg(['mean', 'std', 'count'])
print("--- GROUPED AGGREGATION RESULTS ---")
print(grouped_summary.to_string())
highest_mean = grouped_summary['mean'].max()
lowest_mean = grouped_summary['mean'].min()
mean_ratio = highest_mean / lowest_mean

print(f"Ratio of Highest Group Mean to Lowest Group Mean: {mean_ratio:.2f}")